# Obtaining Data from USDA Local Food Directory

In [8]:
import requests
import geopandas as gpd
from io import BytesIO 
import urllib.parse
import pandas as pd

In [7]:
# Obtaining shapefiles for target Texas counties 
# 1. Define the endpoint
base_url = "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/State_County/MapServer/1/query"

# 2. Build parameters (Note the addition of returnGeometry)
query_params = {
    "where": "BASENAME IN ('Dallas', 'Tarrant', 'Collin', 'Harris', 'Travis', 'Bexar') AND STATE='48'", 
    "outFields": "*",
    "returnGeometry": "true",  # CRITICAL: Forces the API to include the polygons
    "f": "geojson"
}

encoded_params = urllib.parse.urlencode(query_params)
request_url = f"{base_url}?{encoded_params}"

# 3. Fetch the data
response = requests.get(request_url)
response.raise_for_status()
data = response.json()

# 4. Ensure we actually got features back
if 'features' not in data or len(data['features']) == 0:
    raise ValueError("The API returned an empty dataset. Check your 'where' clause.")

# 5. Load into GeoPandas
target_counties = gpd.GeoDataFrame.from_features(data["features"])

# 6. CRITICAL FIX: Explicitly set the active geometry column
# We check to ensure the column exists first so it doesn't throw a confusing error
if 'geometry' in target_counties.columns:
    target_counties = target_counties.set_geometry('geometry')
else:
    raise KeyError("The API did not return a 'geometry' column. The API endpoint may not support GeoJSON export natively.")

# 7. Set the coordinate reference system
target_counties.set_crs(epsg=4326, inplace=True)

# Save it to disk
target_counties.to_file('texas_counties.geojson', driver='GeoJSON')
print("Successfully downloaded and saved spatial data.")

Successfully downloaded and saved spatial data.


In [ ]:
# Examining the type of response received from the USDA API
API_key = '4DBWYY5kkS'
response = requests.get(f'https://www.usdalocalfoodportal.com/api/csa/?apikey={API_key}&state=tx', headers=headers)
response.raise_for_status()

# Print the first 250 characters of the response
print(response.text[:250])

In [17]:
# 1. Load the county shapefile
counties = gpd.read_file('../fooddesertproject/texas_counties.geojson')

# 2. Make the API request
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
API_key = '4DBWYY5kkS'
response = requests.get(f'https://www.usdalocalfoodportal.com/api/csa/?apikey={API_key}&state=tx', headers=headers)
response.raise_for_status()

# 2. Parse JSON into a standard Pandas DataFrame
data = response.json()

# If the list of points is nested under a key (e.g., data['results']), adjust this accordingly:
df = pd.DataFrame(data)

# 1. Unpack the nested JSON column
# .tolist() extracts the column of dictionaries into a standard Python list, 
# and passing that to pd.DataFrame() instantly expands the keys into individual columns.
expanded_df = pd.DataFrame(df['data'].tolist())

# 2. Convert the expanded DataFrame into a GeoDataFrame
raw_points = gpd.GeoDataFrame(
    expanded_df, 
    geometry=gpd.points_from_xy(expanded_df['location_x'], expanded_df['location_y']),
    crs="EPSG:4326"  # Standard API GPS coordinates
)

# 3. Load your target county boundaries
target_counties = gpd.read_file('texas_counties.geojson')

# 4. Clip the points (ensure CRS matches)
if raw_points.crs != target_counties.crs:
    raw_points = raw_points.to_crs(target_counties.crs)

clipped_points = gpd.clip(raw_points, target_county)

# 5. Save the final clipped data
clipped_points.to_file('usdaFarms.gpkg', driver='GPKG')

In [24]:
clipped_points['listing_name'].value_counts()

listing_name
Green Gate Farms                        2
Johnson's Backyard Garden               1
Tecolote Farm                           1
Munkebo Farm                            1
Plant It Forward                        1
Fisher Family Farm & Ranch              1
Booster Club Foods/eOrganic Products    1
GreenTexasFarms                         1
Squeezepenny Sustainable Farm           1
Name: count, dtype: int64

In [29]:
# dropping the one duplicate 
clipped_points = clipped_points.drop([1])
clipped_points.head(20)